<center>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/SN_web_lightmode.png" width="300">
</center>


<h1>Analysis of Global COVID-19 Pandemic Data</h1>

Estimated time needed: **90** minutes



## Overview:

In this final project, you will apply your knowledge of data analysis by completing a series of hands-on tasks in this lab notebook. The lab consists of 10 tasks, where you will write and execute code to demonstrate your skills.

While working through the lab, remember to save all your code and outputs, then download the Jupyter Notebook, as you will need to submit the completed notebook for the **Final Project: Submission and Evaluation**.

If you need to refresh your memories about specific coding details, you may refer to previous hands-on labs for code examples.


In [52]:
# This lab requires 'httr' and 'rvest'packages, which are already pre-loaded into this lab environment.
# However, if you are working on your local RStudio, please uncomment the below codes and install the packages.

#install.packages("httr")
#install.packages("rvest")

In [53]:
library(httr)
library(rvest)

Note: if you can import above libraries, please use install.packages() to install them first.


## TASK 1: Get a `COVID-19 pandemic` Wiki page using HTTP request


First, let's write a function to use HTTP request to get a public COVID-19 Wiki page.

Before you write the function, you can open this public page from this 

URL https://en.wikipedia.org/w/index.php?title=Template:COVID-19_testing_by_country using a web browser.

The goal of task 1 is to get the html page using HTTP request (`httr` library)


In [57]:

get_wiki_covid19_page <- function() {

  # Our target COVID-19 wiki page URL is: https://en.wikipedia.org/w/index.php?title=Template:COVID-19_testing_by_country  
  # Which has two parts: 
    # 1) base URL `https://en.wikipedia.org/w/index.php  
    # 2) URL parameter: `title=Template:COVID-19_testing_by_country`, seperated by question mark ?
    
  # Wiki page base
  wiki_base_url <- "https://en.wikipedia.org/w/index.php"
    
  # You will need to create a List which has an element called `title` to specify which page you want to get from Wiki
  # in our case, it will be `Template:COVID-19_testing_by_country`
 data<- list(title="Template:COVID-19_testing_by_country")
  # - Use the `GET` function in httr library with a `url` argument and a `query` arugment to get a HTTP response
    data1 <- GET(
        url = wiki_base_url,
        query = data
        
    )
return(data1)
  # Use the `return` function to return the response
}




Call the `get_wiki_covid19_page` function to get a http response with the target html page


In [56]:
# Call the get_wiki_covid19_page function and print the response
get_wiki_covid19_page()

Response [https://en.wikipedia.org/w/index.php?title=Template%3ACOVID-19_testing_by_country]
  Date: 2026-09-27 19:31
  Status: 200
  Content-Type: text/html; charset=UTF-8
  Size: 1 MB
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-fea...
<head>
<meta charset="UTF-8">
<title>Template:COVID-19 testing by country - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-heade...
RLSTATE={"ext.globalCssJs.user.styles":"ready","site.styles":"ready","user.st...
<script>(RLQ=window.RLQ||[]).push(function(){mw.loader.impl(function(){return...
}];});});</script>
<link rel="stylesheet" href="/w/load.php?lang=en&amp;modules=ext.cite.parsoid...
...

## TASK 2: Extract COVID-19 testing data table from the wiki HTML page


On the COVID-19 testing wiki page, you should see a data table `<table>` node contains COVID-19 testing data by country on the page:

<a href="https://cognitiveclass.ai/">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/M5_Final/images/covid-19-by-country.png" width="400" align="center">
</a>

Note the numbers you actually see on your page may be different from above because it is still an on-going pandemic when creating this notebook.

The goal of task 2 is to extract above data table and convert it into a data frame


Now use the `read_html` function in rvest library to get the root html node from response


In [ ]:
# Get the root html node from the http response in task 1 
root_node <- read_html(get_wiki_covid19_page())

Get the tables in the HTML root node using `html_nodes` function.


In [ ]:
# Get the table node from the root html node
root_node <- read_html(get_wiki_covid19_page())
table_node <- html_nodes(root_node, "table")
table_node

Read the specific table from the multiple tables in the `table_node` using the `html_table` function and convert it into dataframe using `as.data.frame`

_Hint:- Please read the `table_node` with index 2(ex:- table_node[2])._


In [ ]:
# Read the table node and convert it into a data frame, and print the data frame for review
covid_table <- html_table(table_node[2])
covid_df <- as.data.frame(covid_table)
print(covid_df)


## TASK 3: Pre-process and export the extracted data frame

The goal of task 3 is to pre-process the extracted data frame from the previous step, and export it as a csv file


Let's get a summary of the data frame


In [58]:
# Print the summary of the data frame
summary(covid_df)

 Country.or.region    Date.a.             Tested            Units.b.        
 Length:173         Length:173         Length:173         Length:173        
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
 Confirmed.cases.   Confirmed..tested.. Tested..population..
 Length:173         Length:173          Length:173          
 Class :character   Class :character    Class :character    
 Mode  :character   Mode  :character    Mode  :character    
 Confirmed..population..     Ref.          
 Length:173              Length:173        
 Class :character        Class :character  
 Mode  :character        Mode  :character  

As you can see from the summary, the columns names are little bit different to understand and some column data types are not correct. For example, the `Tested` column shows as `character`. 

As such, the data frame read from HTML table will need some pre-processing such as removing irrelvant columns, renaming columns, and convert columns into proper data types.


We have prepared a pre-processing function for you to conver the data frame but you can also try to write one by yourself


In [60]:
preprocess_covid_data_frame <- function(data_frame) {
    
    shape <- dim(data_frame)

    # Remove the World row
    data_frame<-data_frame[!(data_frame$`Country.or.region`=="World"),]
    # Remove the last row
    data_frame <- data_frame[1:172, ]
    
    # We dont need the Units and Ref columns, so can be removed
    data_frame["Ref."] <- NULL
    data_frame["Units.b."] <- NULL
    
    # Renaming the columns
    names(data_frame) <- c("country", "date", "tested", "confirmed", "confirmed.tested.ratio", "tested.population.ratio", "confirmed.population.ratio")
    
    # Convert column data types
    data_frame$country <- as.factor(data_frame$country)
    data_frame$date <- as.factor(data_frame$date)
    data_frame$tested <- as.numeric(gsub(",","",data_frame$tested))
    data_frame$confirmed <- as.numeric(gsub(",","",data_frame$confirmed))
    data_frame$'confirmed.tested.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.tested.ratio`))
    data_frame$'tested.population.ratio' <- as.numeric(gsub(",","",data_frame$`tested.population.ratio`))
    data_frame$'confirmed.population.ratio' <- as.numeric(gsub(",","",data_frame$`confirmed.population.ratio`))
    
    return(data_frame)
}


Call the `preprocess_covid_data_frame` function


In [61]:
# call `preprocess_covid_data_frame` function and assign it to a new data frame
processed_covid_df <- preprocess_covid_data_frame(covid_df)
summary(processed_covid_df)

                country             date         tested         
 Afghanistan        :  1   2 Feb 2023 :  6   Min.   :     3880  
 Albania            :  1   1 Feb 2023 :  4   1st Qu.:   512037  
 Algeria            :  1   31 Jan 2023:  4   Median :  3029859  
 Andorra            :  1   1 Mar 2021 :  3   Mean   : 31377219  
 Angola             :  1   23 Jul 2021:  3   3rd Qu.: 12386725  
 Antigua and Barbuda:  1   29 Jan 2023:  3   Max.   :929349291  
 (Other)            :166   (Other)    :149                      
   confirmed        confirmed.tested.ratio tested.population.ratio
 Min.   :       0   Min.   : 0.00          Min.   :   0.006       
 1st Qu.:   37839   1st Qu.: 5.00          1st Qu.:   9.475       
 Median :  281196   Median :10.05          Median :  46.950       
 Mean   : 2508340   Mean   :11.25          Mean   : 175.504       
 3rd Qu.: 1278105   3rd Qu.:15.25          3rd Qu.: 156.500       
 Max.   :90749469   Max.   :46.80          Max.   :3223.000       
           

Get the summary of the processed data frame again


In [62]:
# Print the summary of the processed data frame again
summary(processed_covid_df)

                country             date         tested         
 Afghanistan        :  1   2 Feb 2023 :  6   Min.   :     3880  
 Albania            :  1   1 Feb 2023 :  4   1st Qu.:   512037  
 Algeria            :  1   31 Jan 2023:  4   Median :  3029859  
 Andorra            :  1   1 Mar 2021 :  3   Mean   : 31377219  
 Angola             :  1   23 Jul 2021:  3   3rd Qu.: 12386725  
 Antigua and Barbuda:  1   29 Jan 2023:  3   Max.   :929349291  
 (Other)            :166   (Other)    :149                      
   confirmed        confirmed.tested.ratio tested.population.ratio
 Min.   :       0   Min.   : 0.00          Min.   :   0.006       
 1st Qu.:   37839   1st Qu.: 5.00          1st Qu.:   9.475       
 Median :  281196   Median :10.05          Median :  46.950       
 Mean   : 2508340   Mean   :11.25          Mean   : 175.504       
 3rd Qu.: 1278105   3rd Qu.:15.25          3rd Qu.: 156.500       
 Max.   :90749469   Max.   :46.80          Max.   :3223.000       
           

After pre-processing, you can see the columns and columns names are simplified, and columns types are converted into correct types.


The data frame has following columns:

- **country** - The name of the country
- **date** - Reported date
- **tested** - Total tested cases by the reported date
- **confirmed** - Total confirmed cases by the reported date
- **confirmed.tested.ratio** - The ratio of confirmed cases to the tested cases
- **tested.population.ratio** - The ratio of tested cases to the population of the country
- **confirmed.population.ratio** - The ratio of confirmed cases to the population of the country


OK, we can call `write.csv()` function to save the csv file into a file. 


In [63]:
# Export the data frame to a csv file
write.csv(processed_covid_df, "covid.csv")

Note for IBM Waston Studio, there is no traditional "hard disk" associated with a R workspace.

Even if you call `write.csv()` method to save the data frame as a csv file, it won't be shown in IBM Cloud Object Storage asset UI automatically.

However, you may still check if the `covid.csv` exists using following code snippet:


In [64]:
# Get working directory
wd <- getwd()
# Get exported 
file_path <- paste(wd, sep="", "/covid.csv")
# File path
print(file_path)
file.exists(file_path)

[1] "/resources/RP0101EN/labs/M5/covid.csv"

[1] TRUE

In [65]:
## Download a sample csv file
# covid_csv_file <- download.file("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0101EN-Coursera/v2/dataset/covid.csv", destfile="covid.csv")
# covid_data_frame_csv <- read.csv("covid.csv", header=TRUE, sep=",")

## TASK 4: Get a subset of the extracted data frame

The goal of task 4 is to get the 5th to 10th rows from the data frame with only `country` and `confirmed` columns selected


In [66]:
# Read covid_data_frame_csv from the csv file
dataa <- processed_covid_df[5:10, c("country","confirmed")]
print(dataa)
# Get the 5th to 10th rows, with two "country" "confirmed" columns


               country confirmed
5               Angola     20981
6  Antigua and Barbuda       832
7            Argentina   9060495
8              Armenia    422963
9            Australia  10112229
10             Austria   5789991

## TASK 5: Calculate worldwide COVID testing positive ratio

The goal of task 5 is to get the total confirmed and tested cases worldwide, and try to figure the overall positive ratio using `confirmed cases / tested cases`


In [67]:
# Get the total confirmed cases worldwide
total_confirmed_people <- sum(processed_covid_df$confirmed)
# Get the total tested cases worldwide
total_tested_people <- sum(processed_covid_df$tested)
# Get the positive ratio (confirmed / tested)
positive_ratio <- total_confirmed_people / total_tested_people
print(total_confirmed_people)
print(total_tested_people)
print(positive_ratio)

[1] 431434555[1] 5396881644[1] 0.07994145

## TASK 6: Get a country list which reported their testing data 

The goal of task 6 is to get a catalog or sorted list of countries who have reported their COVID-19 testing data


In [68]:
# Get the `country` column
country_list <- processed_covid_df$country
# Check its class (should be character)
class(country_list)
# Sort the countries AtoZ
country_AtoZ <- sort(country_list)
# Sort the countries ZtoA
country_ZtoA <- sort(country_list, decrasing=TRUE)
# Print the sorted ZtoA list
print(country_ZtoA)

[1] "factor"

  [1] Afghanistan            Albania                Algeria               
  [4] Andorra                Angola                 Antigua and Barbuda   
  [7] Argentina              Armenia                Australia             
 [10] Austria                Azerbaijan             Bahamas               
 [13] Bahrain                Bangladesh             Barbados              
 [16] Belarus                Belgium                Belize                
 [19] Benin                  Bhutan                 Bolivia               
 [22] Bosnia and Herzegovina Botswana               Brazil                
 [25] Brunei                 Bulgaria               Burkina Faso          
 [28] Burundi                Cambodia               Cameroon              
 [31] Canada                 Chad                   Chile                 
 [34] China[c]               Colombia               Costa Rica            
 [37] Croatia                Cuba                   Cyprus[d]             
 [40] Czechia            

## TASK 7: Identify countries names with a specific pattern

The goal of task 7 is using a regular expression to find any countires start with `United`


In [69]:
# Use a regular expression `United.+` to find matches
matched_country <- grep("United.+", processed_covid_df$country, value = TRUE)
# Print the matched country names
print(matched_country)

[1] "United Arab Emirates" "United Kingdom"       "United States"       

## TASK 8: Pick two countries you are interested, and then review their testing data

The goal of task 8 is to compare the COVID-19 test data between two countires, you will need to select two rows from the dataframe, and select `country`, `confirmed`, `confirmed-population-ratio` columns


In [70]:
# Select a subset (should be only one row) of data frame based on a selected country name and columns
austria_data <- subset(processed_covid_df, country=="Austria", select = c(country, confirmed, confirmed.population.ratio))
print(austria_data)
View(austria_data)
# Select a subset (should be only one row) of data frame based on a selected country name and columns
germany_data <- subset(processed_covid_df, country=="Germany", select= c(country,confirmed, confirmed.population.ratio))
print(germany_data)
View(germany_data)

   country confirmed confirmed.population.ratio
10 Austria   5789991                         65

,country,confirmed,confirmed.population.ratio
,<fct>,<dbl>,<dbl>
10,Austria,5789991,65


   country confirmed confirmed.population.ratio
60 Germany   3733519                        4.5

,country,confirmed,confirmed.population.ratio
,<fct>,<dbl>,<dbl>
60,Germany,3733519,4.5


## TASK 9: Compare which one of the selected countries has a larger ratio of confirmed cases to population

The goal of task 9 is to find out which country you have selected before has larger ratio of confirmed cases to population, which may indicate that country has higher COVID-19 infection risk


In [71]:
# Use if-else statement
# if (check which confirmed.population value is greater) {
#    print()
# } else {
#    print()
# }
if (austria_data$confirmed.population.ratio > germany_data$confirmed.population.ratio){
    print("Austria has higher Germany to population ratio")
} else {
    print("Germany has higher Austria to population ratio")
}

[1] "Austria has higher Germany to population ratio"

## TASK 10: Find countries with confirmed to population ratio rate less than a threshold

The goal of task 10 is to find out which countries have the confirmed to population ratio less than 1%, it may indicate the risk of those countries are relatively low


In [72]:
# Get a subset of any countries with `confirmed.population.ratio` less than the threshold
low_virus_countries <- subset(processed_covid_df, confirmed.population.ratio < 0.01)
print(low_virus_countries)

        country        date    tested confirmed confirmed.tested.ratio
28      Burundi  5 Jan 2021     90019       884                  0.980
34     China[c] 31 Jul 2020 160000000     87655                  0.055
89         Laos  1 Mar 2021    114030        45                  0.039
119 North Korea 25 Nov 2020     16914         0                  0.000
156    Tanzania 18 Nov 2020      3880       509                 13.100
    tested.population.ratio confirmed.population.ratio
28                   0.7600                    0.00740
34                  11.1000                    0.00610
89                   1.6000                    0.00063
119                  0.0660                    0.00000
156                  0.0065                    0.00085